# Project 2 - Global Space Economy Analysis

## Notebook 03: Industry Analysis using SQL

**Objectives**

- Connect to the SQLite database
- Explore the financial dataset
- Analyze companies using SQL
- Compare industries and financial performance

In [8]:
import pandas as pd
import sqlite3

In [12]:
conn = sqlite3.connect("../data/space_economy.db")

In [13]:
pd.read_sql(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table';
    """,
    conn
)

,name
0,companies
1,financials


In [14]:
pd.read_sql(
    """
    SELECT *
    FROM financials
    LIMIT 5;
    """,
    conn
)

,Company,Ticker,Currency,Market Cap Local,Revenue Local,Net Income Local,Current Price Local,52 Week High Local,52 Week Low Local,Beta,Trailing PE,Forward PE,Profit Margin,Data Date,USD Exchange Rate,FX Date,Market Cap USD,Revenue USD,Net Income USD
0,SpaceX,SPCX,USD,1914209435648,19300999168,-9356000256,145.30,225.64,145.07,NaN,NaN,167.635790,-0.44998,2026-07-11,1.000000,2026-07-11,1.914209e+12,1.930100e+10,-9.356000e+09
1,Rocket Lab,RKLB,USD,50635313152,679577984,-182615008,81.04,151.00,37.57,2.553,NaN,2431.443000,-0.26872,2026-07-11,1.000000,2026-07-11,5.063531e+10,6.795780e+08,-1.826150e+08
2,MDA Space,MDA.TO,CAD,6691734528,2098800000,47100000,48.17,67.90,20.85,NaN,63.381577,27.616928,0.02244,2026-07-11,0.706564,2026-07-11,4.728138e+09,1.482936e+09,3.327916e+07
3,Airbus Defence and Space,AIR.PA,EUR,155288616960,72529002496,5014000128,197.26,221.30,157.42,0.880,31.162716,22.960196,0.06913,2026-07-11,1.141944,2026-07-11,1.773308e+11,8.282403e+10,5.725705e+09
4,中国卫星,600118.SS,CNY,106447667200,6270337536,16975960,90.02,127.77,27.27,0.546,9002.000000,630.833860,0.00271,2026-07-11,0.147783,2026-07-11,1.573111e+10,9.266464e+08,2.508750e+06


### 1. Top 10 Companies by Market Capitalization (USD)

This query identifies the ten largest publicly listed companies in the space economy based on market capitalization.

In [26]:
top_market_cap = pd.read_sql(
    """
    SELECT
        Company,
        Ticker,
        ROUND(`Market Cap USD` / 1000000000, 2) AS Market_Cap_USD_Bn
    FROM financials
    WHERE `Market Cap USD` IS NOT NULL
    ORDER BY `Market Cap USD` DESC
    LIMIT 10;
    """,
    conn
)

top_market_cap

,Company,Ticker,Market_Cap_USD_Bn
0,NVIDIA,NVDA,5109.66
1,Alphabet,GOOGL,4358.51
2,Microsoft,MSFT,2860.69
3,Amazon,AMZN,2639.15
4,SpaceX,SPCX,1914.21
5,Broadcom,AVGO,1902.89
6,AMD,AMD,909.70
7,Oracle,ORCL,405.11
8,Palantir,PLTR,303.96
9,RTX,RTX,263.86


### 2. Sector-level Overview

This analysis compares the five major sectors by company count, total market capitalization, average revenue, profitability and risk.

In [17]:
sector_summary = pd.read_sql(
    """
    SELECT
        c.Sector,
        COUNT(*) AS Number_of_Public_Companies,

        ROUND(
            SUM(f.`Market Cap USD`) / 1000000000,
            2
        ) AS Total_Market_Cap_USD_Bn,

        ROUND(
            AVG(f.`Market Cap USD`) / 1000000000,
            2
        ) AS Average_Market_Cap_USD_Bn,

        ROUND(
            AVG(f.`Revenue USD`) / 1000000000,
            2
        ) AS Average_Revenue_USD_Bn,

        ROUND(
            AVG(f.`Profit Margin`) * 100,
            2
        ) AS Average_Profit_Margin_Percent,

        ROUND(
            AVG(f.Beta),
            2
        ) AS Average_Beta

    FROM companies AS c

    JOIN financials AS f
        ON c.Ticker = f.Ticker

    GROUP BY c.Sector

    ORDER BY Total_Market_Cap_USD_Bn DESC;
    """,
    conn
)

sector_summary

,Sector,Number_of_Public_Companies,Total_Market_Cap_USD_Bn,Average_Market_Cap_USD_Bn,Average_Revenue_USD_Bn,Average_Profit_Margin_Percent,Average_Beta
0,Technology,11,18750.81,1704.62,176.02,10.88,1.75
1,Infrastructure,11,2203.88,200.35,10.14,-17.35,1.35
2,Defense,8,826.95,103.37,47.17,7.39,0.37
3,Industrial,10,342.30,34.23,8.83,9.89,0.98
4,Applications,16,143.58,8.97,2.07,-16.14,1.40


### 3. Subsector-level Overview

This analysis explores the performance of each space industry subsector, including launch, satellite manufacturing, semiconductors and earth observation.

In [18]:
subsector_summary = pd.read_sql(
"""
SELECT
    c.Subsector,

    COUNT(*) AS Number_of_Public_Companies,

    ROUND(
        SUM(f.`Market Cap USD`) / 1000000000,
        2
    ) AS Total_Market_Cap_USD_Bn,

    ROUND(
        AVG(f.`Market Cap USD`) / 1000000000,
        2
    ) AS Average_Market_Cap_USD_Bn,

    ROUND(
        AVG(f.`Revenue USD`) / 1000000000,
        2
    ) AS Average_Revenue_USD_Bn,

    ROUND(
        AVG(f.`Profit Margin`) * 100,
        2
    ) AS Average_Profit_Margin_Percent,

    ROUND(
        AVG(f.Beta),
        2
    ) AS Average_Beta

FROM companies AS c

JOIN financials AS f
    ON c.Ticker = f.Ticker

GROUP BY c.Subsector

ORDER BY Total_Market_Cap_USD_Bn DESC;
""",
conn
)

subsector_summary

,Subsector,Number_of_Public_Companies,Total_Market_Cap_USD_Bn,Average_Market_Cap_USD_Bn,Average_Revenue_USD_Bn,Average_Profit_Margin_Percent,Average_Beta
0,Cloud Computing,4,10263.46,2565.87,387.73,28.71,1.39
1,Semiconductors,5,8182.00,1636.40,75.97,29.81,2.01
2,Launch,2,1964.84,982.42,9.99,-35.94,2.55
3,Space Defense,8,826.95,103.37,47.17,7.39,0.37
4,AI & Data Platforms,2,305.35,152.67,2.74,-72.14,1.81
5,Aerospace Components,5,267.50,53.50,14.03,12.74,1.02
6,Satellite Manufacturing,3,197.79,65.93,28.41,3.14,0.71
7,Satellite Communications,8,90.95,11.37,3.23,-22.25,1.36
8,Advanced Materials,5,74.80,14.96,3.63,7.04,0.94
9,Ground Infrastructure,4,36.96,9.24,1.44,6.34,0.96


## 4. Sector Leaders

This analysis identifies the largest listed company within each sector based on market capitalization (USD).

It highlights the market leader in every major segment of the global space economy.

In [20]:
sector_leaders = pd.read_sql(
"""
SELECT
    c.Sector,
    c.Company,
    c.Ticker,
    ROUND(f.`Market Cap USD` / 1000000000,2) AS Market_Cap_USD_Bn

FROM companies c

JOIN financials f
ON c.Ticker = f.Ticker

WHERE f.`Market Cap USD` =
(
    SELECT MAX(f2.`Market Cap USD`)
    FROM financials f2
    JOIN companies c2
    ON c2.Ticker = f2.Ticker

    WHERE c2.Sector = c.Sector
);

""",
conn
)

sector_leaders

,Sector,Company,Ticker,Market_Cap_USD_Bn
0,Infrastructure,SpaceX,SPCX,1914.21
1,Applications,AST SpaceMobile,ASTS,28.46
2,Technology,NVIDIA,NVDA,5109.66
3,Industrial,Parker Hannifin,PH,121.20
4,Defense,RTX,RTX,263.86


## 5. Most Profitable Companies

This analysis identifies the most profitable public companies in the global space economy based on profit margin.

Profit margin measures how much profit a company generates from every dollar of revenue. Higher profit margins generally indicate stronger operational efficiency and pricing power.

In [27]:
most_profitable = pd.read_sql(
"""
SELECT

    Company,
    Ticker,

    ROUND(
        `Profit Margin` * 100,
        2
    ) AS Profit_Margin_Percent,

    ROUND(
        `Revenue USD` / 1000000000,
        2
    ) AS Revenue_USD_Bn

FROM financials

WHERE `Profit Margin` IS NOT NULL
AND `Revenue USD` >= 500000000

ORDER BY `Profit Margin` DESC

LIMIT 10;
""",
conn
)

most_profitable

,Company,Ticker,Profit_Margin_Percent,Revenue_USD_Bn
0,NVIDIA,NVDA,62.97,253.49
1,Palantir,PLTR,43.67,5.22
2,Microsoft,MSFT,39.34,318.27
3,Broadcom,AVGO,38.85,75.46
4,Alphabet,GOOGL,37.92,422.50
5,Hexagon,HEXA-B.ST,37.52,0.56
6,Marvell Technology,MRVL,28.99,8.72
7,Oracle,ORCL,25.37,67.36
8,Kongsberg Defence & Aerospace,KOG.OL,21.99,3.43
9,Parker Hannifin,PH,16.58,20.99


## 6. Highest Revenue Companies

This analysis ranks companies by annual revenue.

Large revenue indicates market scale and commercial maturity, although it does not necessarily imply profitability.

In [23]:
highest_revenue = pd.read_sql(
"""
SELECT

    Company,
    Ticker,

    ROUND(
        `Revenue USD` / 1000000000,
        2
    ) AS Revenue_USD_Bn,

    ROUND(
        `Profit Margin` * 100,
        2
    ) AS Profit_Margin_Percent

FROM financials

WHERE `Revenue USD` IS NOT NULL

ORDER BY `Revenue USD` DESC

LIMIT 10;
""",
conn
)

highest_revenue

,Company,Ticker,Revenue_USD_Bn,Profit_Margin_Percent
0,Amazon,AMZN,742.78,12.22
1,Alphabet,GOOGL,422.50,37.92
2,Microsoft,MSFT,318.27,39.34
3,NVIDIA,NVDA,253.49,62.97
4,Boeing,BA,92.18,2.46
5,RTX,RTX,90.37,8.03
6,Airbus Defence and Space,AIR.PA,82.82,6.91
7,Broadcom,AVGO,75.46,38.85
8,Lockheed Martin,LMT,75.11,6.38
9,Oracle,ORCL,67.36,25.37


## 7. Companies with the Highest Forward Valuation

Forward Price-to-Earnings (Forward PE) reflects investors' expectations of future earnings.

Companies with extremely high Forward PE ratios are often priced based on future growth rather than current profitability.

In [24]:
highest_pe = pd.read_sql(
"""
SELECT

    Company,
    Ticker,

    ROUND(
        `Forward PE`,
        2
    ) AS Forward_PE,

    ROUND(
        `Profit Margin` * 100,
        2
    ) AS Profit_Margin_Percent

FROM financials

WHERE `Forward PE` IS NOT NULL

ORDER BY `Forward PE` DESC

LIMIT 10;
""",
conn
)

highest_pe

,Company,Ticker,Forward_PE,Profit_Margin_Percent
0,Planet Labs,PL,7822.82,-111.17
1,Rocket Lab,RKLB,2431.44,-26.87
2,中国卫星,600118.SS,630.83,0.27
3,Globalstar,GSAT,261.48,-3.09
4,Viasat,VSAT,182.38,-0.73
5,SpaceX,SPCX,167.64,-45.00
6,Palantir,PLTR,60.54,43.67
7,Saab,SAAB-B.ST,54.92,7.86
8,Boeing,BA,53.11,2.46
9,HEICO,HEI,50.93,16.08


## 8. Companies with the Lowest Market Beta

Beta measures how sensitive a company's stock price is relative to the overall market.

A lower beta generally indicates lower market volatility and potentially lower investment risk.

In [25]:
lowest_risk = pd.read_sql(
"""
SELECT

    Company,
    Ticker,

    ROUND(
        Beta,
        2
    ) AS Beta,

    ROUND(
        `Market Cap USD` / 1000000000,
        2
    ) AS Market_Cap_USD_Bn

FROM financials

WHERE Beta IS NOT NULL

ORDER BY Beta ASC

LIMIT 10;
""",
conn
)

lowest_risk

,Company,Ticker,Beta,Market_Cap_USD_Bn
0,Northrop Grumman,NOC,-0.10,76.65
1,BAE Systems,BA.L,-0.07,72.72
2,Saab,SAAB-B.ST,0.01,30.52
3,中科星图,688568.SS,0.06,5.01
4,Lockheed Martin,LMT,0.11,120.64
5,Eutelsat,ETL.PA,0.15,3.17
6,Kongsberg Defence & Aerospace,KOG.OL,0.22,26.96
7,RTX,RTX,0.30,263.86
8,Syensqo,SYENS.BR,0.45,7.69
9,中国卫星,600118.SS,0.55,15.73


# Conclusions and Next Steps

This notebook explored the financial structure of the global space economy using SQL queries on the integrated company database.

The analysis covered:

- Top public companies ranked by market capitalization
- Sector-level financial performance
- Subsector-level financial performance
- Market leaders across major sectors
- Companies with the highest profitability
- Companies generating the highest revenue
- Companies with the highest forward valuation (Forward P/E)
- Companies with the lowest market risk (Beta)

Overall, the results show that the space economy extends far beyond launch companies. Technology, semiconductors, cloud computing, aerospace, defense and satellite-related businesses all play important roles within the industry's value chain.

These findings provide the quantitative foundation for the next notebook, which will investigate a key investment question:

**Which industries and companies are likely to benefit the most from the long-term growth of the global space economy?**